# Case study -- Analysis of an _ex vivo_ experiment 
## Introduction
### Case-study presentation


### Experimental details


## Preparation of the data
### Importing libraries and modules


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import pingouin as pg

In [ ]:
import platform

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Pingouin:", pg.__version__)

In [ ]:
import warnings
warnings.simplefilter(action='ignore')

# Set the maximum number of rows to display
# pd.set_option('display.max_rows', 15)

### Loading raw data


In [ ]:
# Import the raw data
data_raw = pd.read_csv('../data/ifng_data.csv', sep=";", na_values='na')

print("First rows of the raw dataset:")
print(data_raw.head())

In [ ]:
print("Information about the raw dataset:")
print(data_raw.info())

In [ ]:
# Mapping the dose to the group label
map_groups = {
    'a': 0,
    'b': 2,
    'c': 10,
    'd': 50,
    'e': 250}

# Create a copy of the raw dataset (also a good practice ;-)
data = data_raw.copy()

# Add a new column to the dataset
data['dose_µg'] = data['group'].map(map_groups)

# Create a new variable based on the group id and animal number
data['animal_id'] = data['group'] + '|' + data['animal_id'].apply(str)

# Remove the group label
data.drop(columns='group', inplace=True)

# Encode the stimulation condition for the ANOVA
data['stim_id'] = data['stim'] + '|' + data['conc_µg_ml'].apply(str)

print("First rows of the initial dataset:")
print(data.head())

### Data cleaning


In [ ]:
# Filter for animal No. 5 from group 'b'
ix = data['animal_id'] == 'b|5'
print(data[ix])

In [ ]:
# drop animal b|5 from the whole analysis
data.drop(data[ix].index, inplace=True)

### Sanity check


In [ ]:
print("Information about the initial dataset:")
print(data.info())

In [ ]:
np.random.seed(111)
print("Sample rows with NaN concentration values in the initial dataset:")
print(data[data['conc_µg_ml'].isna()].sample(10))

### Visualization helper function


In [ ]:
def draw_boxnstripplot(
    data: pd.DataFrame,
    x: str = 'dose_µg',
    y: str = 'ifng_pg_ml',
    hue: str = None,
    logscale: bool = True,
    palette: str = 'Accent',
    **kwargs    # Accept all keyword arguments, for example 
                # those passed by FacetGrid
    ):
    """
    Generates a combined boxplot and stripplot to visualize the relationship
    between 'x' and 'y', with distinct colors for different 'hue' categories
    specifically optimized for comparing 'SP' and 'LN' tissues.
    Args:
        data: the Pandas DataFrame containing the data
        x: the name of the column to use for the x-axis 
            (default: 'dose_μg')
        y: the name of the column to use for the y-axis
            (default: 'ifng_pg_ml')
        hue: the name of the column to use for grouping and coloring
            (default: 'tissue', expected to contain 'SP' and 'LN')
        logscale: if y-scale is Log (default: True)
        palette: the color palette to use for the strip plot 
            (default: 'Accent')
    """

    # Plot the box plots
    ax=sns.boxplot(
        x=x, y=y, data=data,
        hue=hue,
        showcaps=False,
        showfliers=False,
        boxprops={
            'linewidth': 2, 'edgecolor': 'black', 'facecolor': 'white'},
        whiskerprops={'linewidth': 2,'color': 'k'},
        capprops={'linewidth': 2, 'color': 'k'},
        medianprops={'linewidth': 2,'color': 'darkgrey'})

    # Plot the data points over the box plots
    sns.stripplot(
        x=x, y=y, data=data,
        hue=hue,
        dodge=True if hue else False,
        palette=palette,
        size=5, alpha=0.85,
        ax=ax)

    # If logscale argument is True
    if logscale: plt.yscale('log')

    sns.despine()

    # Remove duplicated handles and labels from boxplot in legend,
    # keeping the first halves (the ones from box plots)
    handles, labels = ax.get_legend_handles_labels()
    n_items = len(labels) // 2
    if hue:  # Print the legend only is a hue argument is passed
        ax.legend(
            handles[n_items:], labels[n_items:],
            title=hue, fancybox=False)

## QC analysis
### Positive control


In [ ]:
# Create a dataframe with postive control data filtered
ix_pc = data['stim'] == 'PC'
data_pc = data[ix_pc]

print("First rows of the positive control data subset:")
print(data_pc.head())

#### Summary chart of the positive control


In [ ]:

plt.figure(figsize=(3.5, 3))

draw_boxnstripplot(data=data_pc, hue='tissue', palette='Dark2',)

plt.ylim((5, 2e5))
plt.xlabel("Drug dose (µg)")
plt.ylabel("[IFN-γ](pg/mL)")

plt.title("Restimulation with positive control");

#### Summary statistics of the positive control


In [ ]:
print("Summary statistics by drug dose and organ in PC-stimulated samples:")
print(
    data_pc
    .groupby(['tissue','dose_µg'])
    ['ifng_pg_ml']
    .describe(percentiles=[.5])  # Diplay the median only
    .round(3))

#### ANOVA of the positive control


In [ ]:
print("Mixed-design ANOVA on positive control data using Pingouin:")
print(
    data_pc.mixed_anova(
        dv="ifng_pg_ml",
        between='dose_µg',
        within='tissue',
        subject='animal_id',)
    .round(2))

### Systematic overview and preanalysis


In [ ]:
# Filtering data with **all except** PC
data_not_pc = data[~ix_pc]

print("First rows of the dataset subset:")
print(data_not_pc.head(10))

In [ ]:
# Identify unique conditions
unique_conditions = data_not_pc[['animal_id', 'tissue']].drop_duplicates()

new_rows_list = []
for index, row in unique_conditions.iterrows():
    animal = row['animal_id']
    tissue = row['tissue']

    # Get the row for the 'medium' stimulus under this condition
    medium_row = data_not_pc.loc[
        (data_not_pc['animal_id'] == animal)
        & (data_not_pc['tissue'] == tissue)
        & (data_not_pc['stim'] == 'medium')]

    # Create a new row for CMP1 at conc=0 by copying the medium row
    new_row_cmp1 = medium_row.copy()
    new_row_cmp1['stim'] = 'CMP1'
    new_row_cmp1['conc_µg_ml'] = 0.0
    new_row_cmp1['stim_id'] = 'CMP1|0'
    new_rows_list.append(new_row_cmp1)

    # Create a new row for CMP2 at conc=0 by copying the medium row
    new_row_cmp2 = medium_row.copy()
    new_row_cmp2['stim'] = 'CMP2'
    new_row_cmp2['conc_µg_ml'] = 0.0
    new_row_cmp2['stim_id'] = 'CMP2|0'
    new_rows_list.append(new_row_cmp2)

# Concatenate the new rows with the original data, excluding 'medium'
data_without_medium = data_not_pc[data_not_pc['stim'] != 'medium']
data_two_groups = pd.concat(
    [data_without_medium] + new_rows_list, ignore_index=True)

print("Last rows of the final dataset without medium control:")
print(data_two_groups.tail())

In [ ]:

# Create a FacetGrid with drug dose as column and tissue type as row
g = sns.FacetGrid(data_two_groups, col='dose_µg', row='tissue', height=3)

# Populate the FacetGrid using the custom visualization function
g.map_dataframe(
    draw_boxnstripplot,
    x='conc_µg_ml',
    y='ifng_pg_ml',
    hue='stim',
    logscale=False,
    palette='viridis')

# Customize the final figure
g.set_axis_labels(
    x_var="Dose (µg)",
    y_var="[IFN-γ] (pg/mL)")
g.add_legend(title='Compound');

## Assumptions for statistical analysis


### Data exploration on a logarithmic scale


In [ ]:

# Create a FacetGrid with drug dose as column and tissue type as row
g = sns.FacetGrid(data_two_groups, col='tissue', row='stim', height=3)

# Populate the FacetGrid using the custom visualization function
g.map_dataframe(
    draw_boxnstripplot,
    x='conc_µg_ml',
    y='ifng_pg_ml',
    hue='dose_µg',
    logscale=True,
    palette=sns.color_palette('muted', 5))

# Customize the final figure
g.set(ylim=(5, 2e5))
g.set_axis_labels(
    x_var="[stimulus] (µg/mL)",
    y_var="[IFN-γ] (pg/mL)")
g.add_legend(title='Dose (µg)');

### Baseline adjustment


In [ ]:
data_final = data_two_groups.copy()

# Get the value of the medium for each condition and substract from IFN-γ
data_final['adjusted_ifng'] = (
    data_final
    .groupby(['animal_id', 'tissue', 'stim',])
    ['ifng_pg_ml']
    .transform(
        lambda x: x - x[data_two_groups['conc_µg_ml'] == 0].values)
)

# Remove the rows from medium control
data_final.drop(
    data_final[data_final['conc_µg_ml'] == 0].index, inplace=True)

print("First rows of the final dataset:")
print(data_final.head())

### Normality tests


In [ ]:
shapiro_linear = (
    data_final
    .groupby(['tissue', 'stim', 'conc_µg_ml', 'dose_µg'])
    ['adjusted_ifng']
    .apply(pg.normality)
)

print("Last rows of Shapiro-Wilk normality test (baseline-adjusted, \
untransformed)")
print(shapiro_linear.tail(7))

### Applying log transformation


In [ ]:
# Add a new column to the existing dataframe by applying log10
data_final['log_adj_ifng'] = data_final['adjusted_ifng'].apply(np.log10)

# Export the final dataset to a CSV file for later use with ANOVA in R
data_final.to_csv("../data/data_final.csv", index=False)

In [ ]:
shapiro_log = (
    data_final
    .groupby(['tissue', 'stim', 'conc_µg_ml', 'dose_µg'])
    ['log_adj_ifng']
    .apply(pg.normality)
)

print("Shapiro-Wilk normality test (baseline-adjusted, \
log-transformed):")
print(shapiro_log.tail(7))

### Q-Q plots


In [ ]:
#| fig-subcap:
#|   - "baseline-adjusted, untransformed data"
#|   - "baseline-adjusted, log-transformed data"
#| layout-ncol: 2

plt.figure(figsize=(3.5, 3))
pg.qqplot(data_final['adjusted_ifng'])
plt.title("untransformed")
plt.show()

plt.figure(figsize=(3.5, 3))
pg.qqplot(data_final['log_adj_ifng'])
plt.title("log-transformed")
plt.show()

### Homogeneity of variances


In [ ]:
homoscedasticity_linear = data_final.groupby(
    ['tissue', 'stim', 'conc_µg_ml']).apply(
        pg.homoscedasticity, dv='adjusted_ifng', group='dose_µg')

print("Homoscedasticity Levene's test (untransformed):")
print(homoscedasticity_linear.round(3))

In [ ]:
homoscedasticity_log = data_final.groupby(
    ['tissue', 'stim', 'conc_µg_ml']).apply(
        pg.homoscedasticity, dv='log_adj_ifng', group='dose_µg')

print("Homoscedasticity Levene's test (log-transformed):")
print(homoscedasticity_log.round(3))

### Sphericity
### Limitations of mixed-design ANOVA in Python


## Mixed-design ANOVA
### Complex design


### Combining R analysis


# Import the final dataset in R


# A tibble: 6 × 9
# ℹ 1 more variable: log_adj_ifng <dbl>


# Boxplot


### Post-hoc analysis


#### Simple mixed-design interactions


# A tibble: 12 × 9


In [ ]:
print("Mixed-design ANOVA by tissue (Pingouin):")

print(
    data_final
    .groupby(['tissue'])
    .apply(
        pg.mixed_anova,
        dv='log_adj_ifng',
        between='dose_µg',
        within='stim_id',
        subject='animal_id',
        effsize='ng2',
        correction=False)
    .round(3))


#### Differences between R and Pingouin packages


#### Simple main effects analysis


# A tibble: 12 × 9


In [ ]:
print("Simple main effects of drug dose (Pingouin):")

print(
    data_final
    .groupby(['tissue', 'stim_id'])
    .apply(
        pg.anova,
        dv='log_adj_ifng',
        between='dose_µg')
    .round(3))

#### Pairwise comparisons
##### Bonferroni correction


# A tibble: 10 × 12
# ℹ 1 more variable: p.adj.signif <chr>


In [ ]:
import scikit_posthocs as ph

def print_group_ttests(group_data, dv, group_col, tissue, stim_id):
    """
    Performs pairwise t-tests with Bonferroni adjustment and prints 
    the results and significance table for a given group.

    Args:
        group_data (pd.DataFrame): the DataFrame for a single group.
        dv (str): the name of the dependent variable column.
        group_col (str): the name of the grouping column.
        tissue (str): the tissue type (for printing).
        stimid (str): the stimulus condition (for printing).
    """

    print(f"Significance table for tissue={tissue}, stim_id={stim_id}:")

    # Perform pairwise t-tests with Bonferroni adjustment
    bonferroni_results_ph = ph.posthoc_ttest(
        group_data,
        val_col=dv,
        group_col=group_col,
        sort=True,
        pool_sd=False,
        p_adjust='bonferroni')

    print(ph.sign_table(bonferroni_results_ph, lower=False))

In [ ]:
# Filter the DataFrame before grouping
filtered_data = data_final[
    (data_final['tissue'] == 'sp') & (data_final['stim'] == 'CMP2')]

# Group and apply the function to the filtered data
print(
    "Pairwise t-tests (scikit-posthocs; Bonferroni significance table)\n")
filtered_data.groupby(['tissue', 'stim_id']).apply(
    lambda x: print_group_ttests(
        group_data=x,
        dv='log_adj_ifng',
        group_col='dose_µg',
        tissue=x['tissue'].iloc[0],   # Get tissue for the current group
        stim_id=x['stim_id'].iloc[0]  # Get stim for the current group
    ))

##### False discovery rate adjustment


In [ ]:
# We will use the values of the table in the final figure
pwc = (
    data_final
    .groupby(['tissue', 'stim_id'])
    .apply(
        pg.pairwise_tests,
        dv="log_adj_ifng",
        between='dose_µg',
        padjust='fdr_bh'))

print("Multiple t-tests (FDR BH adjusted P values; Pingouin; first rows):")
print(pwc.round(4).head(10))

## Finalized figure


In [ ]:
import math

def asterix(
    ax=None, tissue=None, stim=None, ticks=False, data=None, pwc=None):
    """
    Draws significance lines with asterisks on a box or strip plot
    to indicate statistically significant differences between groups.

    Args:
        ax (matplotlib.axes.Axes, optional): the axes object to draw on.
            Defaults to None.
        tissue (str, optional): the tissue type to filter data by.
            Defaults to None.
        stim (str, optional): the stimulus condition to filter data by.
            Defaults to None.
        ticks (bool, optional): whether to add ticks at the ends of lines.
            Defaults to False.
        data (pd.DataFrame, optional): dataFrame containing the plot data. 
            Defaults to None.
        pwc (pd.DataFrame, optional): dataFrame with pairwise comparison 
            results, including 'A', 'B' (groups), and 'p-corr'. Index 
            should be multi-indexed by (tissue, stim_id). Defaults to None.
    """

    def n_stars(pval):
        """Encodes the number of asterisks corresponding to P value."""
        if   pval <= 1e-4 : return '****'
        elif pval <= 1e-3 : return '***'
        elif pval <= 1e-2 : return '**'
        elif pval <= .05  : return '*'
        else              : return None

    def plot_significant(x1, x2, y, pval, ticklength=0.05):
        """Plots the significance line and asterisks."""

        ax.plot(
            [x1, x1, x2, x2],
            [y - ticklength*ticks, y, y, y - ticklength*ticks],
            lw=1, color='k')
        ax.text(
            (x1 + x2)*.5,
            y - ticklength*1.5,
            n_stars(pval),
            ha='center', va='bottom',
            color='red', fontdict={'size': 7})

    if (ax is None or data is None or pwc is None 
        or tissue is None or stim is None):
        print("Warning: missing required arguments for asterix function")
        return

    # Get unique dose levels for the current tissue and stimulus
    unique_doses = sorted(data['dose_µg'].unique())
    num_doses = len(unique_doses)

    # Scale tick length
    ticklength = 0.04

    # Create a dynamic x_offset mapping
    if num_doses > 1:
        spacing = 0.6 / (num_doses - 1)
        start_offset = -0.3  # Start from the left of the first box
        x_offset = {
            dose: start_offset + i * spacing for i,
            dose in enumerate(unique_doses)}
    else:
        x_offset = {unique_doses[0]: 0} # No offset if only one dose level

    # Main loop: iterate through each concentration 
    # and plot significant differences
    for conc in sorted(data['conc_µg_ml'].unique()):
        # Initial line height
        y = round(
            data[data['conc_µg_ml'] == conc]['log_adj_ifng'].max(), 1)+0.25

        # Extraction of significant differences for 
        # the current tissue and stimulus
        try:
            stim_id = stim + '|' + str(conc)
            pwc_ = pwc.loc[(tissue, stim_id)]
            sign_ = pwc_[pwc_['p-corr'] <= 0.05]

            # Loop over significant pairwise comparisons and plot
            for idx in sign_.index:
                group_a = sign_.loc[idx, 'A']
                group_b = sign_.loc[idx, 'B']
                pval = sign_.loc[idx, 'p-corr']

                if group_a in x_offset and group_b in x_offset:
                    # Adding small epsilon to avoid log10(0)
                    x1 = math.log10(conc + 1e-9) + 1 + x_offset[group_a]
                    x2 = math.log10(conc + 1e-9) + 1 + x_offset[group_b]
                    plot_significant(x1, x2, y, pval)
                    y += 7.5*ticklength  # Increment height for next line
        except KeyError:
            pass  # No significant differences for this (tissue, stim)

In [ ]:

g = sns.FacetGrid(
    data_final,
    col='tissue', row='stim',
    margin_titles=True, height=3)

g.map_dataframe(
    draw_boxnstripplot,
    x='conc_µg_ml', y='log_adj_ifng',
    hue='dose_µg', palette=sns.color_palette('Set2', 5),
    logscale=False)

g.set_axis_labels(
    x_var="[stimulus] (µg/mL)",
    y_var="Adjusted [IFNγ](pg/mL), $\log_{10}$")

g.add_legend(title='Dose (µg)')


# Iterate through the facets to call the asterix function
for (stim_val, tissue_val), ax in g.axes_dict.items():
    data_facet = data_final[
        (data_final['stim'] == stim_val) 
        & (data_final['tissue'] == tissue_val)]
    asterix(ax=ax, tissue=tissue_val, stim=stim_val, 
        data=data_facet, pwc=pwc)

plt.suptitle(
    "3-Way Mixed ANOVA; $F(12.84, 112.37) = 2.522, P = 0.005$",
    fontweight='bold', y=1.04)
plt.annotate(
    text="pwc: unpaired T tests; P adjustment: Benjamini/Hochberg FDR",
    xy=(1.35, -0.25), xycoords='axes fraction',
    ha='right', va='top');

## Expanding the analysis: Combining data sources
### Leveraging animal metadata


In [ ]:
# Load animal metadata from CSV
animals_data_raw = pd.read_csv("../data/animals.csv", sep=';', comment='!')

print("First rows of the raw animal metadata:")
print(animals_data_raw.head())

print()

# Find the comment line at the end of the CSV file
print("Comment in the CSV file:")
with open("../data/animals.csv") as f:
    lines = f.readlines()
    print(lines[-1])

In [ ]:
# Day of termination
day_initiation = 45607

# Copy the raw metadata
animals_data = animals_data_raw.copy()

# Calculate the age of animals at termination
animals_data['age_days'] = (
    day_initiation - animals_data['day_of_birth'].astype(int))
animals_data['animal_id'] = (
    animals_data['group'] + '|' + animals_data ['group_#'].apply(str))

animals_data.drop(
    columns=['cage', 'termination', 'group_#', 'id', 'day_of_birth'],
    inplace=True)

print("First rows of the animals metadata:")
print(animals_data.head())

In [ ]:
print("Summary statistics on animals metadata:")
print(
    animals_data
    .groupby(['group', 'sex'])
    ['age_days']
    .describe()
    .round(3))

In [ ]:

plt.figure(figsize=(3.5, 3))

draw_boxnstripplot(
    data=animals_data,
    x='group',
    y='age_days',
    hue='sex',
    logscale=False)

plt.xlabel("Treatment group")
plt.ylabel("Age at termination (days)")
plt.title("Animal metadata analysis")
plt.tight_layout();

In [ ]:
print("Two-way ANOVA to detect biased distribution of animal ages:")
print(
    pg.anova(
        data=animals_data,
        dv='age_days',
        between=['sex', 'group']).round(3))

In [ ]:
# Merge the ELISA data with the animal metadata
animals_data_merged = data.merge(animals_data, on='animal_id', how='inner')

print("First rows of the merged animals dataset:")
print(animals_data_merged.head())

In [ ]:
print("Row for animal B5 from the animal metadata:")
print(animals_data.loc[animals_data['animal_id'] == 'b|5'])
print()
print("Row for animal B5 from the merged dataset:")
print(animals_data_merged.loc[animals_data_merged['animal_id'] == 'b|5'])

In [ ]:
# Replace the NaN concentration values by 10
animals_data_merged['conc_µg_ml'].fillna(25, inplace=True)

# Drop useless columns
animals_data_merged.drop(columns=['group', 'stim_id'], inplace=True)

# Sort by organs
animals_data_merged.sort_values(
    by='tissue',
    ascending=True,  # lymph node *ln* alphabetically before *sp*
    inplace=True)

In [ ]:

sns.relplot(
    x='age_days', y='ifng_pg_ml', data=animals_data_merged,
    style='sex', hue='conc_µg_ml', size='dose_µg',
    row='tissue', col='stim',
    col_order=['medium', 'CMP1', 'CMP2', 'PC'],
    # row_order=['ln', 'sp'],
    height=3, aspect=1,
    sizes=[1, 10, 50, 100, 250],
    legend='full',
    alpha=.75,
    palette='turbo')

plt.yscale('log');

### Combining flow cytometry data


In [ ]:
cytometry_data_raw = pd.read_csv("../data/cytometry.csv", sep=';', comment='!')

print("First rows of the raw flow cytrometry data:")
print(cytometry_data_raw.head())

In [ ]:
# Copy the raw data
cytometry_data = cytometry_data_raw.copy()
cytometry_data['animal_id'] = (
    cytometry_data['group'] + '|' + cytometry_data['group_#'].apply(str))

cytometry_data.drop(columns='group_#', inplace=True)

In [ ]:
print("Summary statistics on viability (cytometry):")
print(
    cytometry_data
    .groupby('group')
    ['viability_pct']
    .describe()
    .round(3))

In [ ]:

plt.figure(figsize=(3.5, 3))

draw_boxnstripplot(
    data=cytometry_data, x='group', y='viability_pct',
    palette='autumn',
    logscale=False)

plt.xlabel("Treatment group")
plt.ylabel("% viability (flow cytometry)")
plt.ylim((70, 100))
plt.title("Cell viability analysis")
plt.tight_layout();

In [ ]:
cytometry_data_merged = data.merge(cytometry_data, on='animal_id')

cytometry_data_merged.drop(columns='group', inplace=True)

print("Summary statistics on flow cytometry data:")
print(
    cytometry_data_merged
    .groupby(['dose_µg'])
    [['cd4_pct', 'cd8_pct']]
    .describe(percentiles=[.5])
    .round(2)
    .transpose())

In [ ]:
# Remove the '_pct' suffix from column names
cytometry_data_merged.columns = \
cytometry_data_merged.columns.str.rstrip('_pct')

cytometry_data_melt = cytometry_data_merged.melt(
    id_vars=(
        'animal_id', 'tissue', 'stim',
        'conc_µg_ml', 'ifng_pg_ml', 'dose_µg', 'stim_id'),
    value_vars=['cd8', 'cd4'],
    var_name='t_cell_subset',
    value_name='t_cell_subset_pct')

In [ ]:

sns.relplot(
    x='t_cell_subset_pct', y='ifng_pg_ml',
    data=cytometry_data_melt.fillna(25), # Fake concentration for PC
    hue='conc_µg_ml', style='tissue',
    size='dose_µg', sizes=[1, 6, 30, 80, 200],
    row='t_cell_subset', row_order=['cd4', 'cd8'],
    col='stim', col_order=['medium', 'CMP1', 'CMP2', 'PC'],
    height=3,  aspect=1,
    legend='full',
    alpha=.7,
    palette='turbo')
plt.yscale('log');

## Conclusion
